In [1]:
from huggingface_hub import login
login(token="add your keyxxxx")

In [2]:
import os
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
import torch
from torch.utils.data import DataLoader
import os
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import load_dataset,DatasetDict

2026-02-25 12:34:15.546098: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-25 12:34:16.694027: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-25 12:34:16.694075: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-25 12:34:16.778451: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-25 12:34:16.880683: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructio

[2026-02-25 12:35:18,373] [INFO] [real_accelerator.py:219:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/mhossai5/.conda/envs/llm_tor2/compiler_compat/ld: warning: libstdc++.so.6, needed by /home/mhossai5/local/cuda-11.8/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/mhossai5/.conda/envs/llm_tor2/compiler_compat/ld: /home/mhossai5/local/cuda-11.8/lib64/libcufile.so: undefined reference to `std::runtime_error::~runtime_error()@GLIBCXX_3.4'
/home/mhossai5/.conda/envs/llm_tor2/compiler_compat/ld: /home/mhossai5/local/cuda-11.8/lib64/libcufile.so: undefined reference to `__gxx_personality_v0@CXXABI_1.3'
/home/mhossai5/.conda/envs/llm_tor2/compiler_compat/ld: /home/mhossai5/local/cuda-11.8/lib64/libcufile.so: undefined reference to `std::ostream::tellp()@GLIBCXX_3.4'
/home/mhossai5/.conda/envs/llm_tor2/compiler_compat/ld: /home/mhossai5/local/cuda-11.8/lib64/libcufile.so: undefined reference to `std::string::substr(unsigned long, unsigned long) const@GLIBCXX_3.4'
/home/mhossai5/.conda/envs/llm_tor2/compiler_compat/ld: /home/mhossai5/local/cuda-11.8/lib64/libcufile.

In [4]:
# ds = load_dataset("ronig/pdb_sequences", token=True)
ds = load_dataset("csv", data_files="disease.csv")

In [5]:
ds = ds.select_columns(['sequence','disease'])
ds = ds.filter(lambda example: (len(example['sequence']) <= 512) & ( example['disease']=='hiv'))
ds

DatasetDict({
    train: Dataset({
        features: ['sequence', 'disease'],
        num_rows: 430
    })
})

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Path to your saved checkpoint folder
checkpoint_path = "./checkpoints-mistral-3-final"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)

# Load model
model = AutoModelForCausalLM.from_pretrained(checkpoint_path)

# Set model to evaluation mode and move to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(56, 384, padding_idx=0)
    (layers): ModuleList(
      (0-5): 6 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear(in_features=384, out_features=384, bias=False)
          (k_proj): Linear(in_features=384, out_features=384, bias=False)
          (v_proj): Linear(in_features=384, out_features=384, bias=False)
          (o_proj): Linear(in_features=384, out_features=384, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=384, out_features=1024, bias=False)
          (up_proj): Linear(in_features=384, out_features=1024, bias=False)
          (down_proj): Linear(in_features=1024, out_features=384, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm((384,), eps=1e-06)
        (post_attention_layernorm): MistralRMSNorm((384,), eps=1e-06)
      )
    )
    (norm): MistralRMSNorm((384,), eps=1e-06)
    (

In [7]:
def tokenize_function(example):
    return tokenizer(
        ["[BOS]"+i+"[EOS]" for i in example["sequence"]],
        padding="max_length",
        truncation=True,
        max_length=512,
        return_tensors = 'pt',
        add_special_tokens=True  # Adds BOS and EOS
    )

In [8]:
from datasets import load_dataset
# Apply the tokenizer
tokenized_ds = ds.map(tokenize_function, batched=True)

# Split the 'train' set into train/test
split_dataset = tokenized_ds["train"].train_test_split(test_size=0.1, seed=42)

# Wrap it back into a DatasetDict if you want
final_dataset = DatasetDict({
    "train": split_dataset["train"],
    "test": split_dataset["test"]
})



In [9]:
for i in tokenized_ds['train']:
    break

In [10]:
len(i['input_ids'])

512

In [11]:
torch.tensor(i['input_ids'])

tensor([ 1, 18, 22, 18, 13, 13, 18, 20,  9,  3,  3, 22, 21, 12, 17,  9,  3, 20,
        22, 19, 22, 20,  5,  7,  3, 20,  9, 25, 15, 11, 19,  6, 25,  8, 11, 10,
        23, 23, 19, 18,  3, 17,  9, 18,  9, 13, 18, 23, 22,  9, 23, 11, 15, 17,
        12, 21,  9, 18, 17, 15, 15, 17, 19, 18,  8, 18,  9, 19, 22, 20, 13, 21,
        19, 10,  3, 20, 23,  6,  8,  6, 21,  8, 20,  8, 25, 14,  6, 13, 12,  3,
        13, 42,  6,  6, 21,  3, 22, 25,  8,  5,  3, 19, 18, 42,  6, 25, 23,  6,
         8,  6, 22, 23,  9, 20,  9, 21, 18, 22, 43, 20, 20,  3, 20, 21, 12,  9,
        17,  6, 11, 18, 14, 21, 18, 20, 17, 20, 20, 13, 20,  3, 20, 22,  9,  6,
        43, 21, 11, 21,  5, 18,  3, 15,  9, 25, 13, 15, 23, 25, 18, 18, 19, 19,
         9, 12,  3, 17, 38, 13, 11, 25,  6,  9, 20, 38,  7, 19,  9, 22, 17, 20,
        19,  8, 20,  9, 19, 19, 23,  9, 18,  7, 25, 15, 13, 21, 11, 15, 15, 13,
        18, 17,  7,  6, 11,  3, 21, 25,  8,  5, 18, 22, 25,  7,  8, 22, 22, 17,
         9, 21, 19, 13,  6, 13, 12, 19, 

In [12]:
tokenizer.decode(i['input_ids'])

'[BOS] Q V Q L L Q S G A A V T K P G A S V R V S C E A S G Y N I R D Y F I H W W R Q A P G Q G L Q W V G W I N P K T G Q P N N P R Q F Q G R V S L T R H A S W D F D T F S F Y M D L K A L RS D D T A V Y F C A R Q RS D Y W D F D V W G S G T Q V TV S S A S T K G P D I Q M T Q S P S S L S A S V G D TV T I T C Q A N G Y L N W Y Q Q R R G K A P KL L I Y D G S KL E R G V P S R F S G R R W G Q E Y N L T I N N L Q P E D I A T Y F C Q V Y E F V V P G T R L D L K R TV A A P [EOS] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD

In [13]:
import torch
from transformers import (
    AutoModelForCausalLM, 
    TrainingArguments, 
    Trainer,
    DataCollatorForLanguageModeling,     
)

In [14]:
ckpt_path ="./disease-hiv-mistral-finetune-60"
training_args = TrainingArguments(
    output_dir=ckpt_path,                 # where to save checkpoints
    # save_steps=5000,
    save_total_limit=3,                        # keep only last 3 checkpoints
    per_device_train_batch_size=64,            # large batch size for A100
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=4,             # set >1 if you need larger virtual batch
    num_train_epochs= 60,                        # you can adjust based on convergence
    learning_rate=2e-5,                        # standard start for Transformers
    # weight_decay=0.01,                         # helps generalization
    warmup_steps=500,                        # can adjust based on lr scheduler
    logging_dir="./logs",                      # tensorboard logs
    logging_steps=10,                         # log every 500 steps
    # fp16=True,                                 # mixed precision for speed/memory
    bf16=True,                                 # mixed precision for speed/memory
    report_to="wandb",                   # or "wandb"
    load_best_model_at_end=True,              # optional, keep best model
    # metric_for_best_model="accuracy",         # change to your metric
    save_strategy="epoch",
    eval_strategy="epoch"
)

In [15]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
data_collator

DataCollatorForLanguageModeling(tokenizer=PreTrainedTokenizerFast(name_or_path='./checkpoints-mistral-3-final', vocab_size=56, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '[BOS]', 'eos_token': '[EOS]', 'pad_token': '[PAD]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[BOS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[EOS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
), mlm=False, mlm_probability=0.15, mask_replace_prob=0.8, random_replace_prob=0.1, pad_to_multiple_of=None, tf_experimental_compile=False, return_tensors='pt', seed=None)

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_dataset['train'],
    eval_dataset=final_dataset['test'],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()

/scratch/local/ipykernel_97287/1435521183.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: hossainstudy7 (hossainstudy7-freelancer). Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,No log,2.439470
2,No log,2.426870
3,No log,2.405724
4,No log,2.378637
5,2.426100,2.351134
6,2.426100,2.325190
7,2.426100,2.299654
8,2.426100,2.275442
9,2.426100,2.251843
10,2.311700,2.229267


TrainOutput(global_step=60, training_loss=2.1194864908854165, metrics={'train_runtime': 23.5596, 'train_samples_per_second': 985.587, 'train_steps_per_second': 2.547, 'total_flos': 379604084981760.0, 'train_loss': 2.1194864908854165, 'epoch': 30.0})

In [17]:
trainer.save_model(ckpt_path+'-final')

In [18]:
ckpt_path

'./disease-hiv-mistral-finetune-60'

In [15]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Path to your saved checkpoint folder
checkpoint_path = ckpt_path+'-final'


# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)

# Load model
model = AutoModelForCausalLM.from_pretrained(checkpoint_path)

# Set model to evaluation mode and move to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(56, 384, padding_idx=0)
    (layers): ModuleList(
      (0-5): 6 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear(in_features=384, out_features=384, bias=False)
          (k_proj): Linear(in_features=384, out_features=384, bias=False)
          (v_proj): Linear(in_features=384, out_features=384, bias=False)
          (o_proj): Linear(in_features=384, out_features=384, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=384, out_features=1024, bias=False)
          (up_proj): Linear(in_features=384, out_features=1024, bias=False)
          (down_proj): Linear(in_features=1024, out_features=384, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm((384,), eps=1e-06)
        (post_attention_layernorm): MistralRMSNorm((384,), eps=1e-06)
      )
    )
    (norm): MistralRMSNorm((384,), eps=1e-06)
    (

# Inference

In [16]:
def generate_protein_sequences(
    prompts,
    model,
    tokenizer,
    max_new_tokens=170,
    temperature=1.0,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.1,
    num_return_sequences=10,
    device="cuda"
):
    """
    prompts: list[str]  → batch prompts
    returns: list[str]  → generated protein sequences
    """

    model.eval()
    model.to(device)

    if isinstance(prompts, str):
        prompts = [prompts]

    # batch tokenize
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            do_sample=True,
            repetition_penalty=repetition_penalty,
            num_return_sequences=num_return_sequences,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    # remove spaces for protein tokens
    decoded = [seq.replace(" ", "") for seq in decoded]

    return decoded

In [17]:
import Levenshtein
from itertools import combinations

def novelty_score(generated, training_set):
    training_set = set(training_set)
    novel = [seq for seq in generated if seq not in training_set]
    return len(novel) / len(generated), novel

def diversity_score(sequences):
    """
    Calculates diversity using the formula:
    (2 / (n * (n - 1))) * sum of distances between all pairs.
    Distance = normalized Levenshtein.
    """
    n = len(sequences)
    if n < 2:
        return 0.0  # not enough sequences to compute diversity

    total_dist = 0.0
    for x, y in combinations(sequences, 2):
        dist = Levenshtein.distance(x, y)
        norm_dist = dist / max(len(x), len(y))  # normalize to [0, 1]
        total_dist += norm_dist

    diversity = (2 / (n * (n - 1))) * total_dist
    return diversity

def jaccard_similarity(seq1, seq2, k=3):
    set1 = set(seq1[i:i+k] for i in range(len(seq1)-k+1))
    set2 = set(seq2[i:i+k] for i in range(len(seq2)-k+1))
    return len(set1 & set2) / len(set1 | set2)

# Compare all generated sequences using Jaccard
def average_jaccard(sequences, k=3):
    scores = []
    for a, b in combinations(sequences, 2):
        scores.append(jaccard_similarity(a, b, k))
    return sum(scores) / len(scores) if scores else 1.0

def unique_score(sequences):
    unique = set(sequences)
    return len(unique) / len(sequences),len(sequences)- len(unique) 

In [21]:
prompt = "[BOS]"
sequences = []
for _ in range(10):
    generated = generate_protein_sequences(prompt,model = model, tokenizer =tokenizer )
    sequences.extend(generated)
    # print(f'Length: {len(generated)}',"Generated Protein Sequence:\n", generated)

In [22]:
import pandas as pd
df = pd.DataFrame(sequences,columns = ['sequence'])
df

,sequence
0,VQLVESGGGVVQPGRSLRLSCAASGFTFSNYGMHWVRQAPDKGLEW...
1,QVRLVQSGTEMKKPGASVKVSCRTSGYNFIDNYLHWVRQAPGQDLQ...
2,QEQLVESGGGLVKPEGSLKLSCTASGFSFSNKAVMCWVRQAPGKGL...
3,QEQLVESGGDLVKPGASLTLTCTASGVSFSYRCWICWVRQAPGKGL...
4,EVQLVESGGALIQPGGSLRLSCAVSGFTFSIYWMHWVRQAPGKGPV...
...,...
95,QQQLVESGGGLVKPGASLTLTCTTSGFSFSNNHYMCWVRQAPGKGL...
96,ASVKVSCKASGYNFTNYAIPWVRQAPGQGLEWMGWIDSYFGGTRYA...
97,GGSLRLSCAASGFTFSSYAMHWVRQAPGKGLEWVAVISFDVGNKNY...
98,QVQLVQSGAEVKKPGSSVKVSCKASGDTFSTHSISWVRQAPGQGLE...


In [23]:
df.to_csv(checkpoint_path+'.csv',index = False)

In [25]:
# Example usage
train_sequences = ds['train'].to_pandas()['sequence'].tolist()  # list of known/training sequences
generated_sequences = list(set(df['sequence'].tolist()))
novelty, novel_seq = novelty_score(generated_sequences, train_sequences)
print(f"Novelty: {novelty:.3f}")

Novelty: 1.000


In [26]:
uni_score, no_of_duplicates = unique_score(df['sequence'].tolist())
print(f"Unique: {uni_score:.3f}")
print('duplicates: ',no_of_duplicates)

Unique: 1.000
duplicates:  0


In [27]:
dev_score = diversity_score(generated_sequences)
print(f"Diversity: {dev_score:.3f}")

Diversity: 0.705


In [28]:
jaccard_score = average_jaccard(generated_sequences)
print(f"Average Jaccard similarity (3-mers): {jaccard_score:.3f}")


Average Jaccard similarity (3-mers): 0.069


# 1000 samples

In [18]:
prompt = "[BOS]"
sequences = []
for _ in range(100):
    generated = generate_protein_sequences(prompt,model = model, tokenizer =tokenizer )
    sequences.extend(generated)
    # print(f'Length: {len(generated)}',"Generated Protein Sequence:\n", generated)

In [19]:
import pandas as pd
df = pd.DataFrame(sequences,columns = ['sequence'])
df

,sequence
0,QVQLKESGPGLVKPSETLSLSCSVSGAPINTAYFWAWIRHAPGKAL...
1,GGSLRLSCAASGFIFRNYCMHWIRQVPGKGLEWVALILQDGSLKLY...
2,QVQLQESGPGLVKPSGTLSLTCAVSGGSISSSNWWSWVRQPPGKGL...
3,ASVKVSCKASGYTFTTYDINWVRQATGQGLEWMGWMNPNSGHTGYA...
4,VQLVESGGGVVQPGRSLRLSCATSGFSLNDYGMHWVRQAPGKGLEW...
...,...
995,ASVKVSCKVSGSSFTDLSIHWVRQAPEKGLEWMGGMYPRSGFIQYA...
996,QLQLQESGPGLVKPSETLSLTCTISAGDISSYSSAWSRIRQSPSRR...
997,SQTLSLTCTVSGGSIRSDVYYWAWIRQPPGKGLEWLGHIYHSGKTI...
998,EQLVESGGDLVKPEGTLTLTCKASGIDFNTYDYMCWVRQAPGKGLE...


In [20]:
df.to_csv(checkpoint_path+'-1000.csv',index = False)

In [21]:
# Example usage
train_sequences = ds['train'].to_pandas()['sequence'].tolist()  # list of known/training sequences
generated_sequences = list(set(df['sequence'].tolist()))
novelty, novel_seq = novelty_score(generated_sequences, train_sequences)
print(f"Novelty: {novelty:.3f}")

Novelty: 1.000


In [22]:
uni_score, no_of_duplicates = unique_score(df['sequence'].tolist())
print(f"Unique: {uni_score:.3f}")
print('duplicates: ',no_of_duplicates)

Unique: 1.000
duplicates:  0


In [23]:
dev_score = diversity_score(generated_sequences)
print(f"Diversity: {dev_score:.3f}")

Diversity: 0.694


In [24]:
jaccard_score = average_jaccard(generated_sequences)
print(f"Average Jaccard similarity (3-mers): {jaccard_score:.3f}")


Average Jaccard similarity (3-mers): 0.077


In [25]:
import torch
import math
import torch.nn.functional as F

def calculate_perplexity(
    sequences,
    model,
    tokenizer,
    device="cuda",
    batch_size=8
):
    """
    sequences: list[str]  → protein sequences (validation set)
    returns: float → perplexity
    """

    model.eval()
    model.to(device)

    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for i in range(0, len(sequences), batch_size):
            batch = sequences[i:i+batch_size]

            inputs = tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True
            ).to(device)

            input_ids = inputs["input_ids"]
            attention_mask = inputs["attention_mask"]

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits

            # Shift for next-token prediction
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = input_ids[:, 1:].contiguous()

            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=tokenizer.pad_token_id,
                reduction="sum"
            )

            total_loss += loss.item()
            total_tokens += (shift_labels != tokenizer.pad_token_id).sum().item()

    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)

    return perplexity

ppl = calculate_perplexity(
    generated_sequences,
    model,
    tokenizer
)

print("Perplexity:", round(ppl,3))

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Perplexity: 12.477
